# 02 · Clean listings and derive vehicle age

## Goal
Produce a local analysis table and reconcile every row removed. Retain positive asking prices while marking extreme values for review.

## Snapshot findings
Cleaning retained **393,985 of 426,880** listings (92.3%). All exclusions were zero prices. Among retained listings, **99.68%** have usable age, **99.42%** have an odometer reading before the review flag is applied, and **61.57%** have a condition label.

## Setup

In [1]:
from pathlib import Path
import sys

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "listings.py").is_file()), None)
if root is None:
    raise RuntimeError("Start Jupyter from the repository or its notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from IPython.display import display

from src.listings import (
    RAW, PROCESSED, MIN_MODEL_YEAR, LOW_PRICE_REVIEW, HIGH_PRICE_REVIEW,
    MILEAGE_REVIEW, read_source, clean_listings,
)
raw = read_source(root / RAW)


## Cleaning decisions

1. Parse numeric fields and ISO posting dates. Trim and lowercase category labels. Leave missing values missing.
2. Keep the newest posting for each repeated known ID. Ties use the first occurrence; rows without IDs cannot be matched and stay in the cohort.
3. Exclude missing, unparseable, or non-positive asking prices. Flag very low and high positive prices without removing those records.
4. Require an integral model year from the earliest plausible year through one year after the latest observed posting year. Age is the year shown in the listing minus model year. Invalid dates, invalid years, and negative ages leave age missing. Set negative mileage to missing and flag very large mileage.
5. Use available records for each question. Save a local table without IDs, URLs, exact locations, or free text.

In [2]:
clean, audit, quality = clean_listings(raw)
print(f"Review flags: below {LOW_PRICE_REVIEW:,} USD or above {HIGH_PRICE_REVIEW:,} USD; "
      f"above {MILEAGE_REVIEW:,} mi. Minimum model year: {MIN_MODEL_YEAR}.")
display(audit.round(2))
display(quality.to_frame())


Review flags: below 500 USD or above 150,000 USD; above 1,000,000 mi. Minimum model year: 1886.


,listings,removed,numeric_prices,min_usd,p25_usd,median_usd,p75_usd,max_usd
stage,,,,,,,,
Parsed source,426880,0,426880,0.0,5900.0,13950.0,26485.75,3.736929e+09
Unique known listing IDs (missing IDs kept),426880,0,426880,0.0,5900.0,13950.0,26485.75,3.736929e+09
Positive asking price,393985,32895,393985,1.0,7000.0,15000.0,27590.00,3.736929e+09


,listings
Source rows with no listing ID,0
Duplicate listing ID rows removed,0
Missing / unparseable price after deduplication,0
Non-positive price after deduplication,32895
Missing / unparseable posting date among retained rows,62
Invalid populated model year among retained rows,0
Negative calculated age among retained rows,106
Negative odometer among retained rows,0
Odometer above review threshold among retained rows,466
Price below low review threshold among retained rows,9199


## Check coverage by question

In [3]:
fields = ["vehicle_age", "model_year", "odometer", "condition", "manufacturer",
          "fuel", "type", "state"]
available = clean[fields].notna().sum()
display(available.to_frame("available").assign(
    available_pct=(100 * available / len(clean)).round(2)
))
print(f"Retained {len(clean):,} / {len(raw):,} records ({len(clean) / len(raw):.1%}).")


,available,available_pct
vehicle_age,392706,99.68
model_year,392812,99.70
odometer,391695,99.42
condition,242596,61.57
manufacturer,377800,95.89
fuel,391391,99.34
type,308053,78.19
state,393985,100.00


Retained 393,985 / 426,880 records (92.3%).


## Write the local table

In [4]:
destination = root / PROCESSED
destination.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(destination, index=False)
print(f"Wrote {len(clean):,} records to {PROCESSED.as_posix()}.")


Wrote 393,985 records to data/processed/listings.csv.


## Next steps
Use the local output in [03 · Price, age, and mileage](03_price_age_mileage.ipynb) and [04 · Categories and states](04_categories_and_states.ipynb).